# Customer Onboarding & AML Screening

**Project 02 — AI Testing Portfolio**

A Python program that decides whether a new bank customer can open an account, with automated tests that verify its rules.

**What it does**
- Screens the customer's name against a sanctions list
- Checks eligibility: minimum age and a valid ID
- Flags politically exposed persons (PEP) and high-risk countries for Enhanced Due Diligence (EDD)
- Returns one decision: `Approve`, `EDD`, or `Reject` with a reason

> All names, lists, and country codes in this notebook are fictional and used for training only.

## 1. Configuration

`TODAY` is fixed so the tests give the same result on any day. Using the real system date would make the tests flaky.

In [1]:
TODAY = "2026-09-26"
SANCTIONS_LIST = ["Omar Khalid", "Salem Nasser", "Tariq Mansour"]
HIGH_RISK_COUNTRIES = ["XA", "XB"]

## 2. Customer record

Each customer is stored as a dictionary: one record, with a named field for each piece of data.

In [2]:
customer = {
    "name": "Yousif Hassan",
    "age": 35,
    "nationality": "SD",
    "is_pep": False,
    "id_expiry": "2027-09-01",
}

print(customer["name"])
print(customer.get("email", "Not provided"))

Yousif Hassan
Not provided


## 3. Eligibility rules

- The customer must be **18 or older**
- The ID must **not be expired**

Dates are compared as `YYYY-MM-DD` text, which sorts correctly because the year comes first.

In [3]:
def check_eligibility(customer):
    if customer["age"] < 18:
        return "Reject: under 18"
    if customer["id_expiry"] < TODAY:
        return "Reject: ID expired"
    return "Eligible"

## 4. Sanctions screening

A plain comparison misses names typed with different letter case or extra spaces. That is a **false negative**: a sanctioned person passes the check.

`normalize` removes extra spaces (at the ends and between words) and converts the name to lower case before comparing.

In [4]:
def normalize(name):
    return " ".join(name.split()).lower()


def is_sanctioned(name):
    clean_list = []
    for s in SANCTIONS_LIST:
        clean_list.append(normalize(s))
    return normalize(name) in clean_list

## 5. Onboarding decision

The order of the checks matters. Sanctions screening runs **first**, so a sanctioned applicant is always recorded with the sanctions reason, even if they would also fail another rule. Compliance needs to know about every sanctions match.

In [5]:
def get_decision(customer):
    if is_sanctioned(customer["name"]):
        return "Reject: sanctions match"

    eligibility = check_eligibility(customer)
    if eligibility != "Eligible":
        return eligibility

    if customer["is_pep"] or customer["nationality"] in HIGH_RISK_COUNTRIES:
        return "EDD"

    return "Approve"


print(get_decision(customer))

Approve


## 6. Automated tests

`make_customer` is a **test data builder**: it returns a valid customer, and each test changes only the field it is testing.

| Group | Technique | What it proves |
|-------|-----------|----------------|
| Eligibility | Boundary Value Analysis | Age 17/18 and ID expiring yesterday/today/tomorrow |
| Sanctions | Input variations | Letter case and extra spaces do not bypass screening |
| Decision | Equivalence partitioning | One test for each possible decision |
| Decision | Combination testing | When a customer has two problems, the right rule wins |

In [6]:
def make_customer(name="Ahmed Ali", age=30, nationality="SA",
                  is_pep=False, id_expiry="2027-01-01"):
    return {"name": name, "age": age, "nationality": nationality,
            "is_pep": is_pep, "id_expiry": id_expiry}


# Eligibility: boundary values
assert check_eligibility(make_customer(age=17)) == "Reject: under 18", "17 is below the minimum age"
assert check_eligibility(make_customer(age=18)) == "Eligible", "18 is exactly the minimum age"
assert check_eligibility(make_customer(id_expiry="2026-09-25")) == "Reject: ID expired", "ID expired yesterday"
assert check_eligibility(make_customer(id_expiry="2026-09-26")) == "Eligible", "ID expires today"
assert check_eligibility(make_customer(id_expiry="2026-09-27")) == "Eligible", "ID expires tomorrow"

# Sanctions: input variations
assert is_sanctioned("Salem Nasser") == True, "Exact match"
assert is_sanctioned("salem nasser") == True, "Lower case"
assert is_sanctioned("SALEM NASSER") == True, "Upper case"
assert is_sanctioned("  Salem Nasser  ") == True, "Spaces at both ends"
assert is_sanctioned("Salem    Nasser") == True, "Extra spaces between words"
assert is_sanctioned("Ahmed Ali") == False, "Name not on the list"

# Decision: one test per outcome
assert get_decision(make_customer()) == "Approve"
assert get_decision(make_customer(age=17)) == "Reject: under 18"
assert get_decision(make_customer(id_expiry="2026-01-01")) == "Reject: ID expired"
assert get_decision(make_customer(name="Omar Khalid")) == "Reject: sanctions match"
assert get_decision(make_customer(is_pep=True)) == "EDD"
assert get_decision(make_customer(nationality="XA")) == "EDD"

# Decision: combinations (rule order)
assert get_decision(make_customer(name="Omar Khalid", age=17)) == "Reject: sanctions match", "Sanctions must win over age"
assert get_decision(make_customer(name="Omar Khalid", is_pep=True)) == "Reject: sanctions match", "Sanctions must win over PEP"

print("All tests passed ✅")

All tests passed ✅


## 7. Defects found and mutation testing

**Defect found during testing:** the first version of `normalize` used `strip()`, which only removes spaces at the ends. A name typed with extra spaces between words (`"Salem    Nasser"`) passed the sanctions check. It was fixed by rebuilding the name with single spaces: `" ".join(name.split())`.

**Mutation testing:** the order of the checks in `get_decision` was deliberately swapped so that eligibility ran before sanctions screening.

- The six single-rule tests **still passed**, because each test customer had only one problem.
- The combination test *"Sanctions must win over age"* **failed**, so the defect was caught.

Testing each rule on its own is not enough. Rule-order defects only appear when a customer triggers two rules at once.

## 8. Open questions and known limitations

**Open questions for the business**
- Should an ID that expires **today** be accepted? The code assumes yes (`<`).
- Is **exactly 18** the minimum, or must the customer be over 18? The code assumes 18 is allowed.

**Known limitation**
- The same Arabic name can be written in English in several ways (for example *Salem* and *Salim*, *Omar* and *Umar*). Exact matching does not catch these variations, so `is_sanctioned("Salim Nasser")` returns `False`. Production AML systems use **fuzzy matching** with a similarity score, which trades false negatives against false positives.

## What I learned

- Dictionaries for customer records, and `get` for missing fields
- Text normalization with `split`, `join`, and `lower`
- Functions that call other functions, each responsible for one rule
- Test data builders with default values
- Combination testing catches rule-order defects that single-rule tests miss
- The difference between false negatives and false positives in screening

**Next:** expose this logic as an API with FastAPI, store decisions in a SQLite audit table, and add a Streamlit page with Playwright UI tests.